# TradeSense · Component 03 · Transformer sequence classifier → `models_transformer_v1`
**Role in the system.** The Autoencoder (v2) is an unsupervised anomaly signal; XGBoost is a supervised pattern model on hand-built rolling features. The Transformer is a supervised **sequence** model: it reads the raw scaled 16-bar window of the six Layer 1 features and learns temporal patterns that the hand-built features may miss (for example slowly building volume bursts).

**Order:** freeze a hashed protocol → select one of 4 small architectures on Layer 3 **validation** → train a 3-seed ensemble → compare at a **matched real-market alarm burden** (alarm episodes/day on genuine Layer 1 validation) against the saved XGBoost and Autoencoder scores and a robust-z baseline → **leave-one-scenario-family-out** test → input-gradient saliency → complementarity table (preparation for fusion) → one unlock of the Layer 3 **test** split.

**Rules baked in**
* Fit: Layer 3 v2 **train** rows. Early stopping and architecture selection: Layer 3 **validation**. Test: locked until validation results are saved, then evaluated once for a pre-declared set. Thresholds: genuine Layer 1 validation only.
* The window is exactly 16 bars (current + 15 previous) so that the *same* clean-negative definition as XGBoost applies: a negative bar's whole window is untouched by any injection. Longer windows would see injection echoes in "clean" bars; testing longer context needs a different negative set and is left for later.
* Layer 3 labels describe injection mechanisms in synthetic aggregate data. Results are evidence inside the synthetic world, not proof of detecting real manipulation. Spoofing/layering/liquidity withdrawal are not represented (TGNN/order-book track).
* Nothing frozen is modified; output goes to a staging folder and is renamed only if every test passes.

## 0 · Setup and frozen-input pinning

In [1]:
import os, sys, json, time, math, re, hashlib, shutil, uuid, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats
from IPython.display import display
import torch, torch.nn as nn

warnings.filterwarnings('ignore', category=FutureWarning); warnings.filterwarnings('ignore', category=RuntimeWarning); warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 120); pd.set_option('display.float_format', lambda v: f'{v:,.4g}')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L1 = BASE / 'processed_layer1_final_v1'; L3 = BASE / 'processed_layer3_final_v2'; AE1 = BASE / 'models_autoencoder_v1'; AE2 = BASE / 'models_autoencoder_v2'; XG1 = BASE / 'models_xgboost_v1'
OUT_FINAL = BASE / 'models_transformer_v1'; STAGE = BASE / ('.trf_stage_' + uuid.uuid4().hex[:8])
FROZEN_NAMES = ['processed_layer1_final_v1', 'processed_layer2_final_v3', 'processed_layer2_model_ready_v1', 'processed_layer3_final_v1', 'processed_layer3_final_v2', 'models_autoencoder_v1', 'models_autoencoder_v2', 'models_xgboost_v1']
for p in (L1, L3, AE1, AE2, XG1): assert p.is_dir(), f'required input missing: {p}'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; UTC = lambda: datetime.now(timezone.utc).isoformat(); RUN_STARTED = UTC(); TS = {}; NTHREAD = max(1, os.cpu_count() or 1)
print('base:', BASE, '| device', DEVICE, '| torch', torch.__version__)

def sha256_file(p, chunk=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def pin_dir(d): return {str(p.relative_to(d)): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)} for p in sorted(d.rglob('*')) if p.is_file()}
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (pd.Timestamp, datetime)): return o.isoformat()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def auc_rank(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]; n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0 or np.nanstd(x) == 0: return float('nan')
    r = stats.rankdata(x); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def avg_precision(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]
    if y.sum() == 0: return float('nan')
    o = np.argsort(-x, kind='mergesort'); yy = y[o]; tp = np.cumsum(yy); return float(((tp / np.arange(1, len(yy) + 1)) * yy).sum() / yy.sum())
def wilson(k, n, z=1.96):
    if n == 0: return [float('nan'), float('nan')]
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d; return [max(0.0, c - h), min(1.0, c + h)]

t0 = time.time(); PIN = {n: pin_dir(BASE / n) for n in FROZEN_NAMES if (BASE / n).is_dir()}
print('pinned:', {n: len(v) for n, v in PIN.items()}, f'({time.time()-t0:.0f}s)')

Mounted at /content/drive
base: /content/drive/MyDrive/research datasets | device cuda | torch 2.11.0+cu130
pinned: {'processed_layer1_final_v1': 400, 'processed_layer2_final_v3': 8, 'processed_layer2_model_ready_v1': 27, 'processed_layer3_final_v1': 5, 'processed_layer3_final_v2': 32, 'models_autoencoder_v1': 39, 'models_autoencoder_v2': 47, 'models_xgboost_v1': 22} (137s)


## 1 · FREEZE THE PROTOCOL (before any feature, model or score exists)

In [2]:
FEATURES = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
SYMS = {'BTCUSDT': 'btc', 'ETHUSDT': 'eth'}; SPL = ('train', 'val', 'test'); WARM = 15; N_BARS = 129600; BOUNDS = {'train': (0, 90720), 'val': (90720, 109440), 'test': (109440, 129600)}; SEEDS = [11, 22, 33]
BASE_FAMS = ['pump_and_dump_like', 'fake_breakout_like', 'abnormal_volume_burst', 'trade_flow_imbalance', 'wash_like_aggregate_proxy', 'liquidity_stress_proxy']
PROTOCOL = {
 'protocol_id': 'transformer_protocol_v1', 'statement': 'Written and hashed before any model or score of this notebook exists.',
 'labels': {'positive': 'Layer 3 v2 synthetic_label == 1', 'negative': 'clean: normal_background == 1 AND no non-clean bar in [t-15, t] (identical to models_xgboost_v1)', 'note': 'labels describe injection mechanisms, not verified manipulation'},
 'input': {'window': 16, 'tensor': '16 x 6 robust-scaled Layer 1 features of bars t-15..t (scaling fitted on Layer 1 TRAIN only, identical to the Autoencoder)', 'causality': 'the window ends at t; no later bar is used'},
 'architecture': {'family': 'pre-norm Transformer encoder, 4 heads, feed-forward 2 x d_model, dropout 0.1, learned positional embedding, last-token readout -> 1 logit', 'candidates': [{'d_model': 32, 'layers': 1}, {'d_model': 32, 'layers': 2}, {'d_model': 64, 'layers': 1}, {'d_model': 64, 'layers': 2}], 'scope': 'one pooled model for BTC and ETH (features are symbol-scaled)'},
 'training': {'loss': 'BCE with logits, pos_weight 5', 'optimizer': 'AdamW lr 1e-3 weight_decay 1e-2', 'batch': 512, 'max_epochs': 30, 'patience': 5, 'early_stopping_metric': 'weighted BCE on Layer 3 validation rows', 'gradient_clip': 1.0},
 'data_use': {'fit': 'Layer 3 v2 train rows (positive or clean)', 'early_stopping_and_selection': 'Layer 3 v2 validation rows / events', 'test': 'locked; unlocked once after validation results are saved; declared set only', 'selection_seed': 11, 'final_seeds': SEEDS, 'score': 'mean of the seeds\' logits; probability = sigmoid(score), uncalibrated'},
 'burden': {'definition': 'alarm episodes per day on genuine Layer 1 VALIDATION bars (alarm bars merged when gap <= 5 bars); threshold = lowest on a quantile grid whose episodes/day <= budget', 'budgets': [1, 2, 4], 'primary': 2},
 'selection_rule': {'metric': 'mean over BTC/ETH of Layer 3 validation event recall at the primary budget', 'tolerance': 0.05, 'tie_break': 'among configs within tolerance of the best: smaller (layers, d_model), then higher validation PR-AUC'},
 'comparators_declared': ['transformer_final', 'xgb_final (saved scores of models_xgboost_v1)', 'ae_recommended (saved scores of models_autoencoder_v2)', 'rz (robust-z baseline)'],
 'lofo': {'definition': 'for each of the 6 base families: retrain the selected architecture without any event containing that family (composites containing it removed; early-stopping set too), threshold at the primary budget on genuine Layer 1 validation, recall on the held-out family\'s non-composite events from the train and validation splits only', 'seed': 11, 'comparison': 'saved XGBoost leave-one-family-out result and the Autoencoder on the same events'},
 'saliency': {'method': 'gradient x input on validation positive bars (<= 2000 per symbol), mean absolute value by feature, by lag and by scenario family; approximate, descriptive only (not exact attribution like TreeSHAP)'},
 'complementarity': {'method': 'per event at the primary budget: detected by Transformer / XGBoost / Autoencoder; union recall; descriptive, prepares the fusion stage'},
 'bootstrap': {'B': 200, 'seed': 7}, 'verdict_rule': {'outperforms': 'mean recall at the primary budget exceeds XGBoost by >= 0.05 on BOTH validation and test', 'comparable': 'both differences within +/-0.05', 'lofo_strength': 'mean held-out-family recall >= 0.6 strong, >= 0.3 moderate, else weak'},
 'claims_policy': 'Scores are pattern signals on synthetic mechanisms; no claim of detecting real manipulation or intent.'}
PJ = json.dumps(PROTOCOL, sort_keys=True, separators=(',', ':'), ensure_ascii=False); PROTOCOL_HASH = hashlib.sha256(PJ.encode()).hexdigest()
STAGE.mkdir(parents=True, exist_ok=False)
for sub in ('protocol', 'models', 'scores', 'reports', 'validation', 'preprocessing'): (STAGE / sub).mkdir()
dump_json({**PROTOCOL, 'integrity': {'sha256_of_canonical_json_excluding_this_field': PROTOCOL_HASH, 'frozen_utc': UTC()}}, STAGE / 'protocol' / 'transformer_protocol.json')
TS['protocol_frozen'] = UTC(); print('PROTOCOL FROZEN  sha256 =', PROTOCOL_HASH)

PROTOCOL FROZEN  sha256 = fae02c0a1ea67e09cddd72ed29d8bd6261ee7f9f71ce4831325e7b316485233b


## 2 · Load Layer 1 (genuine) and Layer 3 v2; clean/positive masks identical to XGBoost; saved comparator scores

In [3]:
F1, DT, F3, LB3 = {}, {}, {}, {}
for sym, tag in SYMS.items():
    d = pd.read_parquet(L1 / 'processed_matrices' / f'{sym.lower()}_l1_feature_matrix_90d.parquet'); d['datetime'] = pd.to_datetime(d['datetime'], utc=True); d = d.sort_values('datetime').reset_index(drop=True)
    assert len(d) == N_BARS and all(f in d.columns for f in FEATURES); F1[sym] = d[FEATURES].to_numpy(np.float64); DT[sym] = d['datetime']
    fe = pd.concat([pd.read_parquet(L3 / tag / f'{sp}_features.parquet') for sp in SPL], ignore_index=True); lb = pd.concat([pd.read_parquet(L3 / tag / f'{sp}_labels.parquet') for sp in SPL], ignore_index=True)
    assert list(fe.columns) == ['datetime'] + FEATURES and len(fe) == N_BARS and (pd.to_datetime(fe.datetime, utc=True).reset_index(drop=True) == DT[sym]).all() and (pd.to_datetime(lb.datetime, utc=True).reset_index(drop=True) == DT[sym]).all()
    F3[sym] = fe[FEATURES].to_numpy(np.float64); LB3[sym] = lb
EV3 = pd.concat([pd.read_parquet(L3 / 'events' / f'{t}_events.parquet') for t in SYMS.values()], ignore_index=True); EV3['fams'] = EV3.families_json.apply(json.loads)
def transform(F):
    X = np.asarray(F, np.float64).copy(); X[:, 3] = np.log1p(np.maximum(X[:, 3], 0)); X[:, 5] = np.log1p(np.maximum(X[:, 5], 0)); return X
PREP, Z1, Z3 = {}, {}, {}
for sym, tag in SYMS.items():
    X = transform(F1[sym]); tr = X[WARM:BOUNDS['train'][1]]; med = np.median(tr, axis=0); q75, q25 = np.percentile(tr, [75, 25], axis=0); sc = (q75 - q25) / 1.349
    sc = np.maximum(np.where(sc > 1e-12, sc, np.maximum(tr.std(axis=0), 1e-8)), 1e-8); PREP[sym] = {'median': med, 'scale': sc, 'fit_rows': [WARM, BOUNDS['train'][1]]}
    for d_ in (AE1, AE2):
        pp = json.loads((d_ / 'preprocessing' / f'{tag}_preprocessing.json').read_text()); assert np.allclose(pp['median'], med, rtol=1e-9, atol=1e-12) and np.allclose(pp['scale'], sc, rtol=1e-9, atol=1e-12), 'preprocessing differs from the Autoencoder'
    Z1[sym] = np.clip((X - med) / sc, -20, 20).astype(np.float32); Z3[sym] = np.clip((transform(F3[sym]) - med) / sc, -20, 20).astype(np.float32)
LOOKBACK = 15; FEAT_START = 30; WIN = 16
POS, CLEAN = {}, {}
for sym in SYMS:
    lb = LB3[sym]; POS[sym] = lb.synthetic_label.to_numpy() == 1; nonclean = (lb.normal_background.to_numpy() == 0).astype(np.int64); c = np.concatenate([[0], np.cumsum(nonclean)]); idx = np.arange(N_BARS)
    CLEAN[sym] = ((c[idx + 1] - c[np.maximum(idx - LOOKBACK, 0)]) == 0) & (idx >= FEAT_START); assert not (CLEAN[sym] & POS[sym]).any()
    print(sym, 'positive bars', {sp: int(POS[sym][a:b].sum()) for sp, (a, b) in BOUNDS.items()}, '| clean bars', {sp: int(CLEAN[sym][a:b].sum()) for sp, (a, b) in BOUNDS.items()})
print('events per split:', EV3.groupby(['symbol', 'split']).size().unstack().to_dict('index')); TS['data_loaded'] = UTC()
# ---- comparators: the saved scores of the already published models (nothing is recomputed or retrained) ----
SEL_AE = json.loads((AE2 / 'models' / 'selected_model.json').read_text()); AE_SCORER = SEL_AE['recommended']['scorer']; SC1, SC3, SAVED_THR = {}, {}, {}
xthr = json.loads((XG1 / 'models' / 'thresholds_by_budget.json').read_text()); athr = json.loads((AE2 / 'models' / 'thresholds_by_budget.json').read_text())
for sym, tag in SYMS.items():
    for name, d_, fn in (('xgb_final', XG1, 'xgb_final'), ('ae_recommended', AE2, AE_SCORER.replace('|', '_'))):
        for src, store in (('l1', SC1), ('l3', SC3)):
            df = pd.read_parquet(d_ / 'scores' / f'{tag}_{fn}_scores_{src}.parquet'); assert (pd.to_datetime(df.datetime, utc=True).reset_index(drop=True) == DT[sym]).all(); store[(sym, name)] = df['score'].to_numpy(np.float32)
    SAVED_THR[(sym, 'xgb_final')] = xthr[f'{sym}|xgb_final']; SAVED_THR[(sym, 'ae_recommended')] = athr[f'{sym}|{AE_SCORER}']
    for src, Zd, store in (('l1', Z1, SC1), ('l3', Z3, SC3)): z = np.abs(Zd[sym]).max(axis=1).astype(np.float32); z[:WARM] = np.nan; store[(sym, 'rz')] = z
XGB_LOFO = json.loads((XG1 / 'reports' / 'lofo_report.json').read_text())['lofo']; print('comparators loaded: xgb_final, ae_recommended =', AE_SCORER)

BTCUSDT positive bars {'train': 2682, 'val': 891, 'test': 1672} | clean bars {'train': 85608, 'val': 16929, 'test': 17408}
ETHUSDT positive bars {'train': 2211, 'val': 845, 'test': 1259} | clean bars {'train': 86079, 'val': 16975, 'test': 17821}
events per split: {'BTCUSDT': {'test': 36, 'train': 80, 'val': 30}, 'ETHUSDT': {'test': 36, 'train': 80, 'val': 30}}
comparators loaded: xgb_final, ae_recommended = ctx|cur|fmax


## 4 · Matched alarm burden, evaluation functions (Layer 3 test stays locked)

In [4]:
EP_GAP = 5; VAL = slice(*BOUNDS['val']); BUDGETS = PROTOCOL['burden']['budgets']; PB = PROTOCOL['burden']['primary']
def episodes(alarm, gap=EP_GAP):
    idx = np.flatnonzero(alarm); return 0 if len(idx) == 0 else int(1 + (np.diff(idx) > gap).sum())
PGRID = 1 - np.logspace(-4.6, -0.3, 260)
def thr_for_budget(v, budget):
    v = v[np.isfinite(v)]; days = len(v) / 1440; cand = np.unique(np.quantile(v, PGRID)); ok = [t for t in cand if episodes(v > t) / days <= budget]
    return float(min(ok)) if ok else float(v.max())
THRB = {}
def register(scorers):
    for sym in SYMS:
        for sc in scorers: THRB[(sym, sc)] = {b: thr_for_budget(SC1[(sym, sc)][VAL], b) for b in BUDGETS}
register(['ae_recommended', 'rz', 'xgb_final'])
TEST_STATE = {'unlocked': False, 'unlock_utc': None, 'calls': 0}
def l3_eval(sym, split, scorer, ci=False, events=None):
    if split == 'test': assert TEST_STATE['unlocked'], 'test split is locked'; TEST_STATE['calls'] += 1
    scores = SC3[(sym, scorer)]; ops = {f'b{b}': t for b, t in THRB[(sym, scorer)].items()}; a, b = BOUNDS[split]; lb = LB3[sym]; idx = np.arange(N_BARS); ev = EV3[(EV3.symbol == sym) & (EV3.split == split)].reset_index(drop=True)
    ok = (idx >= a) & (idx < b) & np.isfinite(scores); pos = ok & POS[sym]; neg = ok & CLEAN[sym]; m = pos | neg; y = pos[m].astype(int); s = scores[m]
    res = {'n_events': int(len(ev)), 'n_pos_bars': int(pos.sum()), 'n_neg_bars': int(neg.sum()), 'roc_auc': auc_rank(s, y), 'pr_auc': avg_precision(s, y), 'pr_auc_chance': float(y.mean())}
    if ci:
        rng = np.random.default_rng(PROTOCOL['bootstrap']['seed']); evb = [np.arange(r.start_idx, r.end_idx + 1) for r in ev.itertuples()]; negs = scores[neg]; ba, bp = [], []
        for _ in range(PROTOCOL['bootstrap']['B']):
            pick = rng.integers(0, len(ev), len(ev)); ps = np.concatenate([scores[evb[i]] for i in pick]); ps = ps[np.isfinite(ps)]; yy = np.r_[np.ones(len(ps), int), np.zeros(len(negs), int)]; xx = np.r_[ps, negs]; ba.append(auc_rank(xx, yy)); bp.append(avg_precision(xx, yy))
        res['roc_auc_ci95'] = [float(np.nanpercentile(ba, 2.5)), float(np.nanpercentile(ba, 97.5))]; res['pr_auc_ci95'] = [float(np.nanpercentile(bp, 2.5)), float(np.nanpercentile(bp, 97.5))]
    res['ops'] = {}
    for name, t in ops.items():
        al = scores > t; det = np.array([bool(al[r.start_idx:r.end_idx + 1].any()) for r in ev.itertuples()]); delay = np.array([int(np.argmax(al[r.start_idx:r.end_idx + 1])) if d_ else -1 for r, d_ in zip(ev.itertuples(), det)])
        negal = np.where(neg, al, False); days = max(neg.sum(), 1) / 1440; e2 = ev.assign(detected=det); fam = {k: [int(g.detected.sum()), len(g)] for k, g in e2.groupby('scenario_family')}
        res['ops'][name] = {'threshold': float(t), 'event_recall': float(det.mean()), 'event_recall_ci95': wilson(int(det.sum()), len(det)), 'events_detected': int(det.sum()), 'median_detection_delay_bars': float(np.median(delay[det])) if det.any() else None,
            'clean_negative_episodes_per_day': float(episodes(negal) / days), 'bar_recall': float(al[pos].mean()) if pos.any() else None, 'by_family': fam, 'families_covered': int(sum(1 for v in fam.values() if v[0] > 0)), 'families_total': len(fam),
            'by_severity': {k: [int(g.detected.sum()), len(g)] for k, g in e2.groupby('severity')}, 'by_ood': {str(k): [int(g.detected.sum()), len(g)] for k, g in e2.groupby('is_ood')}}
    res['layer1'] = {}
    for name, t in ops.items():
        v = SC1[(sym, scorer)][a:b]; al = v > t; res['layer1'][name] = {'alarm_bars_per_day': float(al.sum() / ((b - a) / 1440)), 'episodes_per_day': float(episodes(al) / ((b - a) / 1440))}
    return res

## 5 · Model, window gather, training function, and the 4-architecture selection on Layer 3 VALIDATION

In [5]:
OFFS = np.arange(-(WIN - 1), 1)
def windows_at(Z, ii): return Z[np.asarray(ii)[:, None] + OFFS[None, :]]                      # (n, 16, 6): bars t-15..t
class TModel(nn.Module):
    def __init__(s, d, L, h=4):
        super().__init__(); s.inp = nn.Linear(6, d); s.pos = nn.Parameter(torch.zeros(1, WIN, d)); nn.init.normal_(s.pos, std=0.02)
        s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, h, dim_feedforward=2 * d, dropout=0.1, batch_first=True, norm_first=True), L, enable_nested_tensor=False); s.norm = nn.LayerNorm(d); s.head = nn.Linear(d, 1)
    def forward(s, x): return s.head(s.norm(s.enc(s.inp(x) + s.pos)[:, -1])).squeeze(-1)
TP = PROTOCOL['training']; ROWLOG = []; PW = torch.tensor(5.0, device=DEVICE)
def rows_idx(sym, split, excl=None, tag=''):
    a, b = BOUNDS[split]; idx = np.arange(N_BARS); m = (idx >= a) & (idx < b) & (POS[sym] | CLEAN[sym]) & (idx >= FEAT_START)
    if excl is not None: m &= ~excl[sym]
    ii = np.flatnonzero(m); ROWLOG.append({'tag': tag, 'sym': sym, 'split': split, 'min_idx': int(ii.min()), 'max_idx': int(ii.max()), 'n': int(len(ii)), 'n_pos': int(POS[sym][ii].sum())}); return ii, POS[sym][ii].astype(np.float32)
def train_model(cfg, seed, syms=None, excl=None, tag=''):
    syms = syms or list(SYMS); d, L = cfg
    def build(split, suffix):
        parts = [rows_idx(s, split, excl, tag + suffix) for s in syms]; X = np.concatenate([windows_at(Z3[s], p[0]) for s, p in zip(syms, parts)]); y = np.concatenate([p[1] for p in parts]); return torch.tensor(X, device=DEVICE), torch.tensor(y, device=DEVICE)
    Xt, yt = build('train', '|fit'); Xv, yv = build('val', '|es'); torch.manual_seed(seed); np.random.seed(seed); g = torch.Generator().manual_seed(seed)
    m = TModel(d, L).to(DEVICE); opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-2); lossf = nn.BCEWithLogitsLoss(pos_weight=PW); best, bad, best_state, hist = float('inf'), 0, None, []
    for ep in range(TP['max_epochs']):
        m.train(); perm = torch.randperm(len(Xt), generator=g).to(DEVICE); tl = 0.0
        for i in range(0, len(perm), TP['batch']):
            ix = perm[i:i + TP['batch']]; loss = lossf(m(Xt[ix]), yt[ix]); opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), TP['gradient_clip']); opt.step(); tl += loss.item() * len(ix)
        m.eval()
        with torch.no_grad(): ve = float(sum(lossf(m(Xv[i:i + 8192]), yv[i:i + 8192]).item() * len(yv[i:i + 8192]) for i in range(0, len(yv), 8192)) / len(yv))
        hist.append((ep, tl / len(Xt), ve))
        if ve < best - 1e-5: best, bad, best_state, best_ep = ve, 0, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}, ep
        else:
            bad += 1
            if bad >= TP['patience']: break
    m.load_state_dict(best_state); m.eval()
    return m, {'cfg': list(cfg), 'seed': seed, 'n_train': int(len(yt)), 'n_train_pos': int(yt.sum().item()), 'n_es': int(len(yv)), 'n_es_pos': int(yv.sum().item()), 'epochs_run': len(hist), 'best_epoch': best_ep, 'best_val_loss': best}
@torch.no_grad()
def logits(models, Z):
    n = len(Z); out = np.full(n, np.nan, np.float32); ii_all = np.arange(FEAT_START, n)
    for i in range(0, len(ii_all), 16384):
        ii = ii_all[i:i + 16384]; x = torch.tensor(windows_at(Z, ii), device=DEVICE); out[ii] = np.mean([m.eval()(x).cpu().numpy() for m in models], axis=0)
    return out
def score_into(name, models):
    for sym in SYMS:
        for src, store, Zd in (('l1', SC1, Z1), ('l3', SC3, Z3)): store[(sym, name)] = logits(models, Zd[sym])
CFGS = [(32, 1), (32, 2), (64, 1), (64, 2)]; cname = lambda c: f'trf|d{c[0]}|L{c[1]}'; SELROWS, TRAIN_LOG, CFGM = [], [], {}
TS['selection_start'] = UTC(); t0 = time.time()
for c in CFGS:
    m, info = train_model(c, 11, tag=cname(c)); CFGM[c] = [m]; TRAIN_LOG.append({'config': cname(c), **info}); score_into(cname(c), [m]); register([cname(c)]); ev = {s: l3_eval(s, 'val', cname(c)) for s in SYMS}
    SELROWS.append({'config': cname(c), 'd_model': c[0], 'layers': c[1], 'simp': (c[1], c[0]), 'val_recall_btc': ev['BTCUSDT']['ops'][f'b{PB}']['event_recall'], 'val_recall_eth': ev['ETHUSDT']['ops'][f'b{PB}']['event_recall'], 'val_pr_auc': float(np.mean([ev[s]['pr_auc'] for s in SYMS])), 'epochs': info['epochs_run']})
    print(f"{cname(c):12s} val recall@b{PB} BTC {SELROWS[-1]['val_recall_btc']:.2f} ETH {SELROWS[-1]['val_recall_eth']:.2f} PR {SELROWS[-1]['val_pr_auc']:.3f} epochs {info['epochs_run']} ({time.time()-t0:.0f}s)")
SELTAB = pd.DataFrame(SELROWS); SELTAB['primary'] = (SELTAB.val_recall_btc + SELTAB.val_recall_eth) / 2
def select_rule(tab):
    best = tab.primary.max(); tied = tab[tab.primary >= best - PROTOCOL['selection_rule']['tolerance']].copy(); tied['k'] = tied.simp.apply(tuple); tied['npr'] = -tied.val_pr_auc
    return tied.sort_values(['k', 'npr']).iloc[0].config
SELECTED = select_rule(SELTAB); SEL_CFG = [c for c in CFGS if cname(c) == SELECTED][0]; TS['selection_done'] = UTC()
display(SELTAB.drop(columns='simp').set_index('config')); print('SELECTED (validation only):', SELECTED)

trf|d32|L1   val recall@b2 BTC 0.70 ETH 0.63 PR 0.498 epochs 15 (34s)
trf|d32|L2   val recall@b2 BTC 0.67 ETH 0.63 PR 0.521 epochs 10 (63s)
trf|d64|L1   val recall@b2 BTC 0.70 ETH 0.60 PR 0.520 epochs 16 (95s)
trf|d64|L2   val recall@b2 BTC 0.67 ETH 0.57 PR 0.510 epochs 8 (124s)


,d_model,layers,val_recall_btc,val_recall_eth,val_pr_auc,epochs,primary
config,,,,,,,
trf|d32|L1,32,1,0.7,0.6333,0.4979,15,0.6667
trf|d32|L2,32,2,0.6667,0.6333,0.5212,10,0.65
trf|d64|L1,64,1,0.7,0.6,0.5201,16,0.65
trf|d64|L2,64,2,0.6667,0.5667,0.5103,8,0.6167


SELECTED (validation only): trf|d32|L1


## 6 · Final 3-seed ensemble and VALIDATION results (saved before the test split is unlocked)

In [6]:
FINAL, FINAL_LOG = [], []; TS['final_train_start'] = UTC()
for sd in SEEDS: m, info = train_model(SEL_CFG, sd, tag='final'); FINAL.append(m); FINAL_LOG.append(info)
score_into('transformer_final', FINAL); register(['transformer_final']); TS['final_train_end'] = UTC(); print('final ensemble trained:', FINAL_LOG)
DECL = ['transformer_final', 'xgb_final', 'ae_recommended', 'rz']
TS['val_eval_start'] = UTC(); RES_VAL = {f'{s}|{c}': l3_eval(s, 'val', c, ci=True) for s in SYMS for c in DECL}
dump_json({'protocol_sha256': PROTOCOL_HASH, 'selected': SELECTED, 'declared_set': DECL, 'thresholds': {f'{k[0]}|{k[1]}': v for k, v in THRB.items()}, 'val_results': RES_VAL, 'saved_utc': UTC()}, STAGE / 'reports' / 'validation_results.json'); TS['val_saved'] = UTC()
def comp_table(R, op):
    rows_ = {}
    for k, v in R.items():
        o = v['ops'][op]; rows_[k] = {'ROC': round(v['roc_auc'], 3), 'PR': round(v['pr_auc'], 3), 'chance': round(v['pr_auc_chance'], 3), 'recall': f"{o['events_detected']}/{v['n_events']} ({o['event_recall_ci95'][0]:.2f}-{o['event_recall_ci95'][1]:.2f})",
                      'delay': o['median_detection_delay_bars'], 'L1 ep/d': round(v['layer1'][op]['episodes_per_day'], 2), 'L3clean ep/d': round(o['clean_negative_episodes_per_day'], 2), 'fam': f"{o['families_covered']}/{o['families_total']}"}
    return pd.DataFrame(rows_).T
display(comp_table(RES_VAL, f'b{PB}'))

final ensemble trained: [{'cfg': [32, 1], 'seed': 11, 'n_train': 176580, 'n_train_pos': 4893, 'n_es': 35640, 'n_es_pos': 1736, 'epochs_run': 15, 'best_epoch': 9, 'best_val_loss': 0.42415890667574974}, {'cfg': [32, 1], 'seed': 22, 'n_train': 176580, 'n_train_pos': 4893, 'n_es': 35640, 'n_es_pos': 1736, 'epochs_run': 14, 'best_epoch': 8, 'best_val_loss': 0.41827576409285316}, {'cfg': [32, 1], 'seed': 33, 'n_train': 176580, 'n_train_pos': 4893, 'n_es': 35640, 'n_es_pos': 1736, 'epochs_run': 17, 'best_epoch': 11, 'best_val_loss': 0.4161558471143446}]


,ROC,PR,chance,recall,delay,L1 ep/d,L3clean ep/d,fam
BTCUSDT|transformer_final,0.837,0.516,0.05,21/30 (0.52-0.83),4,1.92,1.79,7/7
BTCUSDT|xgb_final,0.839,0.468,0.05,22/30 (0.56-0.86),5.5,2,2.13,7/7
BTCUSDT|ae_recommended,0.717,0.181,0.05,16/30 (0.36-0.70),2,2,2.04,7/7
BTCUSDT|rz,0.614,0.086,0.05,0/30 (0.00-0.11),None,0,0,0/7
ETHUSDT|transformer_final,0.877,0.532,0.047,19/30 (0.46-0.78),7,2,2.21,6/7
ETHUSDT|xgb_final,0.854,0.536,0.047,19/30 (0.46-0.78),5,1.92,1.87,6/7
ETHUSDT|ae_recommended,0.727,0.156,0.047,11/30 (0.22-0.54),6,2,2.04,5/7
ETHUSDT|rz,0.628,0.078,0.047,0/30 (0.00-0.11),None,0,0,0/7


## 7 · Leave-one-scenario-family-out: does the sequence model generalise to a mechanism it never saw?

In [7]:
base = EV3[EV3.composite == 0]; L2N = {}
for r in base.itertuples(): L2N.setdefault(r.fams[0], set()).add(r.scenario_family)
assert all(len(v) == 1 for v in L2N.values()) and len(L2N) == 6, L2N
L2N = {k: next(iter(v)) for k, v in L2N.items()}; EV3['fam_names'] = EV3.fams.apply(lambda l: sorted(L2N[x] for x in l))
def event_recall(sym, scorer, evs, op=f'b{PB}'):
    al = SC3[(sym, scorer)] > THRB[(sym, scorer)][int(op[1:])]; return int(sum(bool(al[r.start_idx:r.end_idx + 1].any()) for r in evs.itertuples())), len(evs)
LOFO, LOFO_LOG = {}, []; TS['lofo_start'] = UTC()
for fam in BASE_FAMS:
    excl = {}
    for sym in SYMS:
        m_ = np.zeros(N_BARS, bool)
        for r in EV3[(EV3.symbol == sym) & EV3.fam_names.apply(lambda l: fam in l)].itertuples(): m_[r.start_idx:r.end_idx + 1] = True
        excl[sym] = m_
    n0 = len(ROWLOG); mf, info = train_model(SEL_CFG, 11, excl=excl, tag='lofo|' + fam); nm = 'lofo_' + fam; score_into(nm, [mf]); register([nm]); fl = ROWLOG[n0:]
    for sym in SYMS:
        a, bnd = BOUNDS['train']; exp_pos = int(POS[sym][a:bnd].sum() - (POS[sym] & excl[sym])[a:bnd].sum()); got = sum(r['n_pos'] for r in fl if r['sym'] == sym and r['tag'].endswith('|fit'))
        LOFO_LOG.append({'family': fam, 'sym': sym, 'expected_train_pos_bars_without_family': exp_pos, 'train_pos_bars_in_fold_rows': int(got), 'excluded_pos_bars_total': int((POS[sym] & excl[sym]).sum())})
    tot = {'trf_fold': [0, 0], 'ae': [0, 0], 'trf_final_seen': [0, 0], 'xgb_final_seen': [0, 0]}
    for sym in SYMS:
        evs = EV3[(EV3.symbol == sym) & (EV3.composite == 0) & (EV3.scenario_family == fam) & EV3.split.isin(['train', 'val'])]
        for key, sc in (('trf_fold', nm), ('ae', 'ae_recommended'), ('trf_final_seen', 'transformer_final'), ('xgb_final_seen', 'xgb_final')):
            k, n = event_recall(sym, sc, evs); tot[key][0] += k; tot[key][1] += n
    LOFO[fam] = {**tot, 'xgb_fold_saved': XGB_LOFO[fam]['xgb_fold'], 'thr_b%d' % PB: {s: THRB[(s, nm)][PB] for s in SYMS}}
    for sym in SYMS: del SC1[(sym, nm)], SC3[(sym, nm)], THRB[(sym, nm)]
    print(f"held out {fam:28s} transformer {tot['trf_fold'][0]}/{tot['trf_fold'][1]} | XGB (saved) {LOFO[fam]['xgb_fold_saved'][0]}/{LOFO[fam]['xgb_fold_saved'][1]} | AE {tot['ae'][0]}/{tot['ae'][1]} | transformer family seen {tot['trf_final_seen'][0]}/{tot['trf_final_seen'][1]}")
rr_ = lambda k, src=LOFO: float(np.mean([src[f][k][0] / max(src[f][k][1], 1) for f in BASE_FAMS]))
LOFO_MEAN = {'transformer_unseen': rr_('trf_fold'), 'xgb_unseen_saved': rr_('xgb_fold_saved'), 'ae': rr_('ae'), 'transformer_seen': rr_('trf_final_seen')}
LOFO_STRENGTH = 'strong' if LOFO_MEAN['transformer_unseen'] >= 0.6 else ('moderate' if LOFO_MEAN['transformer_unseen'] >= 0.3 else 'weak'); print('macro mean recall:', {k: round(v, 3) for k, v in LOFO_MEAN.items()}, '->', LOFO_STRENGTH); TS['lofo_done'] = UTC()

held out pump_and_dump_like           transformer 31/34 | XGB (saved) 20/34 | AE 10/34 | transformer family seen 32/34
held out fake_breakout_like           transformer 27/34 | XGB (saved) 18/34 | AE 9/34 | transformer family seen 28/34
held out abnormal_volume_burst        transformer 7/34 | XGB (saved) 5/34 | AE 6/34 | transformer family seen 14/34
held out trade_flow_imbalance         transformer 13/32 | XGB (saved) 7/32 | AE 2/32 | transformer family seen 21/32
held out wash_like_aggregate_proxy    transformer 10/32 | XGB (saved) 2/32 | AE 11/32 | transformer family seen 24/32
held out liquidity_stress_proxy       transformer 28/32 | XGB (saved) 27/32 | AE 16/32 | transformer family seen 29/32
macro mean recall: {'transformer_unseen': 0.584, 'xgb_unseen_saved': 0.398, 'ae': 0.274, 'transformer_seen': 0.748} -> moderate


## 8 · Gradient x input saliency (approximate) and complementarity with XGBoost / Autoencoder (validation)

In [8]:
rng = np.random.default_rng(0); SAL = {'by_feature': [], 'by_lag': [], 'fam': {f: [] for f in BASE_FAMS}}; a, b = BOUNDS['val']
for sym in SYMS:
    idx = np.arange(N_BARS); pos_i = np.flatnonzero((idx >= a) & (idx < b) & POS[sym] & (idx >= FEAT_START)); pos_i = rng.choice(pos_i, min(2000, len(pos_i)), replace=False)
    x = torch.tensor(windows_at(Z3[sym], pos_i), device=DEVICE, requires_grad=True); out = torch.stack([m.eval()(x) for m in FINAL]).mean(axis=0); out.sum().backward(); gi = (x.grad * x).abs().detach().cpu().numpy()      # (n,16,6)
    SAL['by_feature'].append(gi.sum(axis=1)); SAL['by_lag'].append(gi.sum(axis=2))
    for fam in BASE_FAMS:
        mk = LB3[sym]['label_' + fam].to_numpy()[pos_i] == 1
        if mk.any(): SAL['fam'][fam].append(gi[mk].sum(axis=1))
SALIENCY = {'by_feature': dict(zip(FEATURES, np.concatenate(SAL['by_feature']).mean(axis=0).tolist())), 'by_lag_0_is_current': dict(zip([str(-l) for l in range(WIN - 1, -1, -1)], np.concatenate(SAL['by_lag']).mean(axis=0).tolist())),
            'by_family_feature': {f: dict(zip(FEATURES, np.concatenate(v).mean(axis=0).tolist())) for f, v in SAL['fam'].items() if v}}
print('saliency by feature:', {k: round(v, 3) for k, v in SALIENCY['by_feature'].items()}); display(pd.DataFrame(SALIENCY['by_family_feature']).T.round(3))
print('saliency by lag (0 = current bar):', {k: round(v, 3) for k, v in list(SALIENCY['by_lag_0_is_current'].items())[::-1][:6]}); TS['saliency_done'] = UTC()
def complementarity(split):
    out = {}
    for sym in SYMS:
        evs = EV3[(EV3.symbol == sym) & (EV3.split == split)]; det = {}
        for nm_, sc in (('trf', 'transformer_final'), ('xgb', 'xgb_final'), ('ae', 'ae_recommended')):
            al = SC3[(sym, sc)] > THRB[(sym, sc)][PB]; det[nm_] = np.array([bool(al[r.start_idx:r.end_idx + 1].any()) for r in evs.itertuples()])
        n = len(evs); out[sym] = {'n_events': n, 'trf_only': int((det['trf'] & ~det['xgb']).sum()), 'xgb_only': int((det['xgb'] & ~det['trf']).sum()), 'both': int((det['trf'] & det['xgb']).sum()), 'neither': int((~det['trf'] & ~det['xgb']).sum()),
                                  'recall_trf': float(det['trf'].mean()), 'recall_xgb': float(det['xgb'].mean()), 'recall_ae': float(det['ae'].mean()), 'union_trf_xgb': float((det['trf'] | det['xgb']).mean()), 'union_xgb_ae': float((det['xgb'] | det['ae']).mean()), 'union_all3': float((det['trf'] | det['xgb'] | det['ae']).mean())}
    return out
COMPL = {'val': complementarity('val')}; display(pd.DataFrame(COMPL['val']).T.round(3))

saliency by feature: {'log_returns': 3.034, 'trade_flow_imbalance': 0.916, 'hl_spread_ratio': 5.369, 'rush_order_density': 4.369, 'volume_acceleration': 1.506, 'large_trade_count': 6.322}


,log_returns,trade_flow_imbalance,hl_spread_ratio,rush_order_density,volume_acceleration,large_trade_count
pump_and_dump_like,3.716,1.019,7.26,4.649,1.815,8.771
fake_breakout_like,3.887,0.935,6.902,4.333,1.616,6.018
abnormal_volume_burst,2.861,1.084,5.124,3.675,1.61,8.232
trade_flow_imbalance,2.711,1.077,4.23,4.33,1.267,5.257
wash_like_aggregate_proxy,2.486,0.686,3.872,4.654,1.506,5.424
liquidity_stress_proxy,2.451,0.972,4.655,3.901,1.176,3.645


saliency by lag (0 = current bar): {'0': 3.344, '-1': 1.902, '-2': 1.639, '-3': 1.553, '-4': 1.44, '-5': 1.409}


,n_events,trf_only,xgb_only,both,neither,recall_trf,recall_xgb,recall_ae,union_trf_xgb,union_xgb_ae,union_all3
BTCUSDT,30,1,2,20,7,0.7,0.733,0.533,0.767,0.767,0.8
ETHUSDT,30,3,3,16,8,0.633,0.633,0.367,0.733,0.667,0.767


## 9 · UNLOCK THE TEST SPLIT (once, declared set only)

In [9]:
assert (STAGE / 'reports' / 'validation_results.json').is_file() and not TEST_STATE['unlocked']
TEST_STATE['unlocked'] = True; TEST_STATE['unlock_utc'] = UTC(); TS['test_unlocked'] = TEST_STATE['unlock_utc']
RES_TEST = {f'{s}|{c}': l3_eval(s, 'test', c, ci=True) for s in SYMS for c in DECL}; TEST_STATE['unlocked'] = False; TS['test_evaluated'] = UTC()
COMPL['test'] = complementarity('test'); print('TEST evaluated once for the declared set.'); display(comp_table(RES_TEST, f'b{PB}')); display(pd.DataFrame(COMPL['test']).T.round(3))
DRIFT = {k: {'val_ep_day': RES_VAL[k]['layer1'][f'b{PB}']['episodes_per_day'], 'test_ep_day': RES_TEST[k]['layer1'][f'b{PB}']['episodes_per_day'], 'DRIFT_FLAG': bool(RES_TEST[k]['layer1'][f'b{PB}']['episodes_per_day'] > 3 * PB)} for k in RES_TEST}

TEST evaluated once for the declared set.


,ROC,PR,chance,recall,delay,L1 ep/d,L3clean ep/d,fam
BTCUSDT|transformer_final,0.913,0.711,0.088,27/36 (0.59-0.86),17,1.07,0.99,7/7
BTCUSDT|xgb_final,0.869,0.609,0.088,26/36 (0.56-0.84),11,0.5,0.58,7/7
BTCUSDT|ae_recommended,0.732,0.272,0.088,18/36 (0.34-0.66),10.5,1.64,1.32,7/7
BTCUSDT|rz,0.629,0.139,0.088,0/36 (0.00-0.10),None,0,0,0/7
ETHUSDT|transformer_final,0.845,0.504,0.066,24/36 (0.50-0.80),12,1.43,1.45,7/7
ETHUSDT|xgb_final,0.813,0.506,0.066,24/36 (0.50-0.80),17.5,1.29,1.21,7/7
ETHUSDT|ae_recommended,0.703,0.183,0.066,10/36 (0.16-0.44),20.5,0.79,0.81,6/7
ETHUSDT|rz,0.616,0.101,0.066,0/36 (0.00-0.10),None,0,0,0/7


,n_events,trf_only,xgb_only,both,neither,recall_trf,recall_xgb,recall_ae,union_trf_xgb,union_xgb_ae,union_all3
BTCUSDT,36,6,5,21,4,0.75,0.722,0.5,0.889,0.806,0.917
ETHUSDT,36,4,4,20,8,0.667,0.667,0.278,0.778,0.667,0.778


## 10 · Persist models, scores, reports and manifest (staging only)

In [10]:
(STAGE / 'models' / 'pooled').mkdir(parents=True, exist_ok=True)
for sd, m in zip(SEEDS, FINAL): torch.save({k: v.cpu() for k, v in m.state_dict().items()}, STAGE / 'models' / 'pooled' / f'seed{sd}.pt')
for sym in SYMS:
    dump_json({'features': FEATURES, 'log1p_columns': ['rush_order_density', 'large_trade_count'], 'median': PREP[sym]['median'], 'scale': PREP[sym]['scale'], 'clip': [-20, 20], 'fit_rows': PREP[sym]['fit_rows'], 'source': 'Layer 1 TRAIN only (identical to the Autoencoder)'}, STAGE / 'preprocessing' / f'{SYMS[sym]}_preprocessing.json')
    for src, store in (('l1', SC1), ('l3', SC3)):
        pd.DataFrame({'datetime': DT[sym], 'score': store[(sym, 'transformer_final')].astype(np.float32), 'prob_uncalibrated': (1 / (1 + np.exp(-store[(sym, 'transformer_final')].astype(np.float64)))).astype(np.float32)}).to_parquet(STAGE / 'scores' / f'{SYMS[sym]}_transformer_final_scores_{src}.parquet', index=False)
dump_json({'selected_config': SELECTED, 'd_model': SEL_CFG[0], 'layers': SEL_CFG[1], 'heads': 4, 'window': WIN, 'seeds': SEEDS, 'score': 'mean logit over seeds', 'training': TP, 'thresholds_episodes_per_day_budget': {SYMS[s]: {str(b_): t for b_, t in THRB[(s, 'transformer_final')].items()} for s in SYMS}, 'threshold_source': 'genuine Layer 1 VALIDATION'}, STAGE / 'models' / 'selected_config.json')
dump_json({f'{k[0]}|{k[1]}': {str(b_): t for b_, t in v.items()} for k, v in THRB.items()}, STAGE / 'models' / 'thresholds_by_budget.json')
dump_json({'selection_table': SELTAB.assign(simp=SELTAB.simp.apply(list)).to_dict(orient='records'), 'selected': SELECTED, 'rule': PROTOCOL['selection_rule']}, STAGE / 'reports' / 'selection_table.json')
dump_json({'lofo': LOFO, 'lofo_macro_mean_recall': LOFO_MEAN, 'lofo_strength': LOFO_STRENGTH, 'lofo_row_log': LOFO_LOG}, STAGE / 'reports' / 'lofo_report.json'); dump_json(SALIENCY, STAGE / 'reports' / 'saliency_report.json'); dump_json(COMPL, STAGE / 'reports' / 'complementarity_report.json')
dump_json({'selection_training': TRAIN_LOG, 'final_training': FINAL_LOG, 'row_log': ROWLOG, 'timestamps': TS, 'device': DEVICE, 'torch': torch.__version__}, STAGE / 'reports' / 'training_report.json')
dump_json({'protocol_sha256': PROTOCOL_HASH, 'validation': RES_VAL, 'test': RES_TEST, 'layer1_drift': DRIFT, 'test_unlock_utc': TEST_STATE['unlock_utc'], 'test_evaluation_calls': TEST_STATE['calls']}, STAGE / 'reports' / 'evaluation_report.json')
LIM = ['Layer 3 is synthetic: labels are injection mechanisms in aggregate data. A high score means "resembles an injected mechanism", not "manipulation happened". Spoofing, layering and liquidity withdrawal are not represented.',
       'The window is 16 bars so the clean-negative definition matches XGBoost; longer context (30-60 bars) was not tested and may help slowly building patterns. It needs a stricter negative set.',
       'Supervised sequence models can learn generator-specific signatures. The leave-one-family-out table and the OOD events are the only evidence about unseen mechanisms; both are modest and come from the same generator.',
       'Architecture selection (4 candidates) used 30 validation events per symbol and the same split provided early stopping; validation numbers are slightly optimistic. Test was unlocked once after validation was saved.',
       'Thresholds come from ~13 genuine validation days (13-52 episodes). Layer 1 test drift can change test alarm rates (DRIFT flag).',
       'Scores are uncalibrated logits; conformal calibration is a later stage. Gradient x input saliency is approximate and descriptive. Three seeds; GPU training is not bit-reproducible.',
       'Complementarity numbers are descriptive: the fusion stage must be selected on validation only.']
(STAGE / 'reports' / 'limitations.md').write_text('# Limitations\n\n' + '\n'.join('- ' + l for l in LIM) + '\n', encoding='utf-8')
(STAGE / 'README.md').write_text('\n'.join(['# models_transformer_v1', '', 'Supervised causal Transformer for Component 03. protocol/transformer_protocol.json was frozen (sha256 ' + PROTOCOL_HASH + ') before any model existed.', '',
    '- models/selected_config.json: architecture, thresholds per alarm-episode budget (1/2/4 per day); models/pooled/seed*.pt: state dicts (score = mean logit over seeds)', '- scores/*_transformer_final_scores_{l1,l3}.parquet', '- reports/: selection_table, validation_results, evaluation_report, lofo_report, saliency_report, complementarity_report, training_report, limitations.md', '', 'Read reports/limitations.md before quoting any number.']), encoding='utf-8')
def build_manifest():
    return {str(p.relative_to(STAGE)): {'size': p.stat().st_size, 'sha256': sha256_file(p)} for p in sorted(STAGE.rglob('*')) if p.is_file() and p.name not in ('manifest.json', 'validation_report.json')}
MANIFEST = {'artifact': 'models_transformer_v1', 'protocol_sha256': PROTOCOL_HASH, 'excluded': ['manifest.json', 'validation_report.json'], 'files': build_manifest()}; dump_json(MANIFEST, STAGE / 'manifest.json'); print('manifest entries:', len(MANIFEST['files']))

manifest entries: 21


## 11 · Validation of the NEW artifacts only, publish, and SHARE BLOCK

In [11]:
RES = []
def T(n, name, ok, detail=''):
    RES.append({'test': n, 'name': name, 'pass': bool(ok), 'detail': str(detail)[:300]}); print(('PASS ' if ok else 'FAIL ') + f'T{n:02d} {name}' + ('' if ok else f'  <-- {str(detail)[:200]}'))
req = ['protocol/transformer_protocol.json', 'models/selected_config.json', 'models/thresholds_by_budget.json', 'reports/selection_table.json', 'reports/validation_results.json', 'reports/evaluation_report.json', 'reports/lofo_report.json', 'reports/saliency_report.json', 'reports/complementarity_report.json', 'reports/training_report.json',
       'reports/limitations.md', 'README.md', 'manifest.json'] + [f'preprocessing/{SYMS[s]}_preprocessing.json' for s in SYMS] + [f'scores/{SYMS[s]}_transformer_final_scores_{src}.parquet' for s in SYMS for src in ('l1', 'l3')] + [f'models/pooled/seed{sd}.pt' for sd in SEEDS]
T(1, 'all required files exist', all((STAGE / r).is_file() for r in req), [r for r in req if not (STAGE / r).is_file()][:5])
T(2, 'frozen sources unchanged (Layers 1, 2, 3 v1/v2, Autoencoder v1/v2, XGBoost v1)', {n: pin_dir(BASE / n) for n in PIN} == PIN)
pj = json.loads((STAGE / 'protocol' / 'transformer_protocol.json').read_text()); body = {k: v for k, v in pj.items() if k != 'integrity'}
order = ['protocol_frozen', 'data_loaded', 'selection_start', 'selection_done', 'final_train_end', 'val_saved', 'lofo_done', 'saliency_done', 'test_unlocked', 'test_evaluated']
T(3, 'protocol intact (hash matches) and frozen BEFORE any model; selection, LOFO and saliency only on train/validation; test unlocked last', hashlib.sha256(json.dumps(body, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode()).hexdigest() == PROTOCOL_HASH == pj['integrity']['sha256_of_canonical_json_excluding_this_field'] and all(TS[a_] <= TS[b_] for a_, b_ in zip(order, order[1:])), {k: TS[k] for k in order})
ok4 = True
for sym in SYMS:
    X = np.asarray(F1[sym], float).copy(); X[:, 3] = np.log1p(np.maximum(X[:, 3], 0)); X[:, 5] = np.log1p(np.maximum(X[:, 5], 0)); tr = X[15:90720]; med = np.median(tr, axis=0); iq = np.subtract(*np.percentile(tr, [75, 25], axis=0)) / 1.349; sc_ = np.maximum(np.where(iq > 1e-12, iq, tr.std(axis=0)), 1e-8)
    pp = json.loads((STAGE / 'preprocessing' / f'{SYMS[sym]}_preprocessing.json').read_text()); ok4 &= bool(np.allclose(pp['median'], med, rtol=1e-9) and np.allclose(pp['scale'], sc_, rtol=1e-9) and pp['fit_rows'] == [15, 90720])
T(4, 'preprocessing fitted on Layer 1 TRAIN bars only (independent numpy recompute; identical to the Autoencoder files)', ok4)
ok5 = True
for sym in SYMS:
    for sc in ('xgb_final', 'ae_recommended'):
        for b_, t in THRB[(sym, sc)].items(): ok5 &= bool(np.isclose(t, SAVED_THR[(sym, sc)][str(b_)], rtol=1e-6))
T(5, 'comparator thresholds recomputed here equal the thresholds saved by the XGBoost / Autoencoder notebooks', ok5)
ok6 = True
for sym in SYMS:
    Zs = Z1[sym][:6000].copy(); Zq = Zs.copy(); Zq[3000:] += 5.0; la, lb_ = logits(FINAL, Zs), logits(FINAL, Zq); ok6 &= bool(np.allclose(la[:3000], lb_[:3000], equal_nan=True, atol=1e-5) and not np.allclose(la[3000:], lb_[3000:], equal_nan=True))
T(6, 'the Transformer is causal (changing bars after t leaves the score at <= t unchanged) and reacts to later changes', ok6)
ok7 = True
for sym in SYMS:
    ii = np.flatnonzero(CLEAN[sym]); ok7 &= bool(np.allclose(windows_at(Z1[sym], ii[::7]), windows_at(Z3[sym], ii[::7]), atol=1e-4))
T(7, 'input windows of clean bars are identical in genuine Layer 1 and Layer 3 (no injection leaks into negatives)', ok7)
ok8 = True
for r in ROWLOG:
    if '|fit' in r['tag']: ok8 &= bool(r['min_idx'] >= FEAT_START and r['max_idx'] < BOUNDS['train'][1])
    elif '|es' in r['tag']: ok8 &= bool(r['min_idx'] >= BOUNDS['val'][0] and r['max_idx'] < BOUNDS['val'][1])
    else: ok8 = False
T(8, 'every fitting row is inside Layer 3 TRAIN; every early-stopping row inside VALIDATION; no test row used for fitting', ok8 and len(ROWLOG) > 20)
ok9 = True; rr = np.random.default_rng(5)
for sym in SYMS:
    nb = LB3[sym].normal_background.to_numpy()
    for t in rr.integers(0, N_BARS, 3000): ok9 &= bool(CLEAN[sym][t] == (t >= FEAT_START and all(nb[max(t - 15, 0):t + 1] == 1)))
T(9, 'clean-bar mask independently recomputed (plain loop on 6000 random bars) matches', ok9)
ok10 = True; days_ = (BOUNDS['val'][1] - BOUNDS['val'][0]) / 1440; thj = json.loads((STAGE / 'models' / 'thresholds_by_budget.json').read_text())
for (sym, sc), tb in THRB.items():
    v = SC1[(sym, sc)][BOUNDS['val'][0]:BOUNDS['val'][1]]
    for b_, t in tb.items():
        idx = np.flatnonzero(v > t); n_ep = 0 if len(idx) == 0 else 1 + sum(1 for i in range(1, len(idx)) if idx[i] - idx[i - 1] > 5); ok10 &= bool(n_ep / days_ <= b_ + 1e-9 and np.isclose(thj[f'{sym}|{sc}'][str(b_)], t))
        low = np.unique(np.quantile(v[np.isfinite(v)], PGRID)); low = low[low < t]; ok10 &= bool(all(episodes(v > x) / days_ > b_ for x in low))
T(10, 'every budget threshold recomputed from genuine Layer 1 VALIDATION: episodes/day <= budget and minimal', ok10)
tab = pd.DataFrame(json.loads((STAGE / 'reports' / 'selection_table.json').read_text())['selection_table']); tab['simp'] = tab.simp.apply(tuple)
T(11, 'architecture selection recomputed from the saved validation table alone', select_rule(tab) == SELECTED == json.loads((STAGE / 'models' / 'selected_config.json').read_text())['selected_config'] and len(tab) == 4)
T(12, 'test split evaluated in exactly one unlocked pass, declared set only, after validation results were saved', TEST_STATE['calls'] == len(SYMS) * len(DECL) and set(RES_TEST) == {f'{s}|{c}' for s in SYMS for c in DECL} and TS['val_saved'] < TS['test_unlocked'] < TS['test_evaluated'])
ok13 = True
for sym in SYMS:
    sdf = pd.read_parquet(STAGE / 'scores' / f'{SYMS[sym]}_transformer_final_scores_l3.parquet')['score'].to_numpy()
    for sp, R in (('val', RES_VAL), ('test', RES_TEST)):
        for op, o in R[f'{sym}|transformer_final']['ops'].items():
            evs = EV3[(EV3.symbol == sym) & (EV3.split == sp)]; det = 0
            for r in evs.itertuples():
                hit = False
                for i in range(int(r.start_idx), int(r.end_idx) + 1):
                    if sdf[i] > o['threshold']: hit = True; break
                det += hit
            ok13 &= bool(det == o['events_detected'])
T(13, 'event recall of the final model independently recomputed (plain loop on stored scores), validation and test, every budget', ok13)
mr, _ = train_model(SEL_CFG, SEEDS[0], tag='repro'); s_new = logits([mr], Z3['BTCUSDT']); s_old = logits([FINAL[0]], Z3['BTCUSDT']); vv = slice(BOUNDS['val'][0], BOUNDS['val'][1]); sp_ = stats.spearmanr(s_new[vv], s_old[vv])[0]
T(14, 'retraining the first final seed reproduces its scores (Spearman >= 0.98; GPU training is not bit-exact)', sp_ >= 0.98, f'spearman {sp_:.5f}')
ok15 = True
for sd, m in zip(SEEDS, FINAL):
    m2 = TModel(*SEL_CFG).to(DEVICE); m2.load_state_dict(torch.load(STAGE / 'models' / 'pooled' / f'seed{sd}.pt', map_location=DEVICE)); m2.eval(); a_, b_ = logits([m], Z3['ETHUSDT'][:20000]), logits([m2], Z3['ETHUSDT'][:20000]); ok15 &= bool(np.allclose(a_, b_, equal_nan=True, atol=1e-5))
T(15, 'saved model files reload and reproduce the scores exactly', ok15)
T(16, 'leave-one-family-out folds trained without the held-out family (fold positive-bar counts equal the full count minus the family\'s bars)', all(r['train_pos_bars_in_fold_rows'] == r['expected_train_pos_bars_without_family'] and r['excluded_pos_bars_total'] > 0 for r in LOFO_LOG) and len(LOFO) == 6, [r for r in LOFO_LOG if r['train_pos_bars_in_fold_rows'] != r['expected_train_pos_bars_without_family']][:2])
T(17, 'all stored scores finite after the warm-up', all(np.isfinite(pd.read_parquet(STAGE / 'scores' / f'{SYMS[s]}_transformer_final_scores_{src}.parquet').score.to_numpy()[100:]).all() for s in SYMS for src in ('l1', 'l3')))
man = json.loads((STAGE / 'manifest.json').read_text())['files']; actual = {str(p.relative_to(STAGE)) for p in STAGE.rglob('*') if p.is_file() and p.name not in ('manifest.json', 'validation_report.json')}
T(18, 'manifest lists every file with correct sha256/size', set(man) == actual and all(sha256_file(STAGE / k) == v['sha256'] for k, v in man.items()), sorted(actual ^ set(man))[:5])
for p in STAGE.rglob('*.json'): json.loads(p.read_text())
for p in STAGE.rglob('*.parquet'): pq.read_metadata(p)
T(19, 'all JSON/parquet outputs readable', True)
ALL_PASS = all(r['pass'] for r in RES)
rc = lambda R, sc, op=f'b{PB}': float(np.mean([R[f'{s}|{sc}']['ops'][op]['event_recall'] for s in SYMS]))
D_VAL, D_TEST = rc(RES_VAL, 'transformer_final') - rc(RES_VAL, 'xgb_final'), rc(RES_TEST, 'transformer_final') - rc(RES_TEST, 'xgb_final'); DRIFTED = any(v['DRIFT_FLAG'] for k, v in DRIFT.items() if k.endswith('transformer_final'))
UN_GAIN = float(np.mean([COMPL['test'][s]['union_trf_xgb'] - max(COMPL['test'][s]['recall_trf'], COMPL['test'][s]['recall_xgb']) for s in SYMS]))
if not ALL_PASS: VERDICT = 'BLOCKED'
elif D_VAL >= 0.05 and D_TEST >= 0.05: VERDICT = 'TRANSFORMER OUTPERFORMS XGBoost at matched burden on validation and test (synthetic world)'
elif abs(D_VAL) < 0.05 and abs(D_TEST) < 0.05: VERDICT = 'TRANSFORMER COMPARABLE to XGBoost at matched burden'
elif D_VAL <= -0.05 or D_TEST <= -0.05: VERDICT = 'TRANSFORMER DOES NOT OUTPERFORM XGBoost at matched burden'
else: VERDICT = 'MIXED: Transformer vs XGBoost difference not consistent across validation and test'
VERDICT += f'; unseen-family generalisation: {LOFO_STRENGTH}; union(Transformer, XGBoost) adds {UN_GAIN:+.2f} test recall over the better single model' + ('; Layer 1 TEST BURDEN DRIFT FLAGGED' if DRIFTED else '')
print(f"\n{sum(r['pass'] for r in RES)}/{len(RES)} tests passed | {VERDICT}")
dump_json({'tests': RES, 'verdict': VERDICT, 'protocol_sha256': PROTOCOL_HASH, 'run_started_utc': RUN_STARTED}, STAGE / 'validation' / 'validation_report.json')
assert {n: pin_dir(BASE / n) for n in PIN} == PIN, 'frozen inputs changed during the run'
if ALL_PASS: os.rename(STAGE, OUT_FINAL); where = OUT_FINAL; print('PUBLISHED ->', where)
else: where = STAGE; print('NOT PUBLISHED; staging kept:', where)

def tab_rows(R, op):
    out = {}
    for sym in SYMS:
        for sc in DECL:
            v = R[f'{sym}|{sc}']; o = v['ops'][op]; out[f'{sym[:3]}|{sc}'] = {'ROC': round(v['roc_auc'], 3), 'PR': round(v['pr_auc'], 3), 'chance': round(v['pr_auc_chance'], 3), 'recall': f"{o['events_detected']}/{v['n_events']} ({o['event_recall_ci95'][0]:.2f}-{o['event_recall_ci95'][1]:.2f})", 'delay': o['median_detection_delay_bars'], 'L1 ep/d': round(v['layer1'][op]['episodes_per_day'], 2), 'L3clean ep/d': round(o['clean_negative_episodes_per_day'], 2), 'fam': f"{o['families_covered']}/{o['families_total']}"}
    return pd.DataFrame(out).T
def fam_tab(R, op): return pd.DataFrame({f'{sym[:3]}|{sc}': {f: f'{a_}/{n_}' for f, (a_, n_) in R[f'{sym}|{sc}']['ops'][op]['by_family'].items()} for sym in SYMS for sc in ('transformer_final', 'xgb_final', 'ae_recommended')}).to_string()
print('\n==================== TRANSFORMER — FINAL SHARE BLOCK ====================')
print('1. Output:', where, '| protocol sha256:', PROTOCOL_HASH[:16], '(frozen before any model) | device:', DEVICE, '| window', WIN, 'bars')
print('2. Pipeline tests:', f"{sum(r['pass'] for r in RES)}/{len(RES)}", '| positives/clean bars in TRAIN:', {s: (int(POS[s][:90720].sum()), int(CLEAN[s][:90720].sum())) for s in SYMS})
print('3. Selection on Layer 3 VALIDATION (4 architectures, recall @ budget', PB, 'episodes/day):'); print(SELTAB.drop(columns=['simp']).set_index('config').round(3).to_string()); print('   SELECTED:', SELECTED, '| final epochs:', [f['epochs_run'] for f in FINAL_LOG])
print('4. VALIDATION @ matched budget', PB, 'episodes/day (Layer 3 events, 95% CI):'); print(tab_rows(RES_VAL, f'b{PB}').to_string())
print('5. TEST @ matched budget', PB, '(unlocked once):'); print(tab_rows(RES_TEST, f'b{PB}').to_string())
print('6. Recall at other budgets (mean of BTC/ETH):', {sc: {f'b{b_}': (round(rc(RES_VAL, sc, f'b{b_}'), 3), round(rc(RES_TEST, sc, f'b{b_}'), 3)) for b_ in BUDGETS} for sc in DECL}, '(val, test)')
print('7. Event recall by scenario @ budget', PB, '(VALIDATION):'); print(fam_tab(RES_VAL, f'b{PB}')); print('   (TEST):'); print(fam_tab(RES_TEST, f'b{PB}'))
print('8. Genuine Layer 1 alarm episodes/day (val | test) and drift flag:', {k: (round(v['val_ep_day'], 2), round(v['test_ep_day'], 2), v['DRIFT_FLAG']) for k, v in DRIFT.items() if 'rz' not in k})
print('9. OOD events @ budget', PB, '(TEST):', {f'{s[:3]}|{c}': RES_TEST[f'{s}|{c}']['ops'][f'b{PB}']['by_ood'] for s in SYMS for c in ('transformer_final', 'xgb_final')}); print('   by severity (TEST, transformer):', {s[:3]: RES_TEST[f'{s}|transformer_final']['ops'][f'b{PB}']['by_severity'] for s in SYMS})
print('10. Leave-one-family-out (unseen mechanism; train+val events):'); print(pd.DataFrame({f: {'transformer (unseen)': '%d/%d' % tuple(v['trf_fold']), 'XGBoost (unseen, saved)': '%d/%d' % tuple(v['xgb_fold_saved']), 'AE': '%d/%d' % tuple(v['ae']), 'transformer (seen)': '%d/%d' % tuple(v['trf_final_seen'])} for f, v in LOFO.items()}).T.to_string()); print('    macro mean recall:', {k: round(v, 3) for k, v in LOFO_MEAN.items()}, '->', LOFO_STRENGTH)
print('11. Saliency (gradient x input, validation positives) by feature:', {k: round(v, 3) for k, v in SALIENCY['by_feature'].items()}); print('    by lag (0 = current bar):', {k: round(v, 3) for k, v in list(SALIENCY['by_lag_0_is_current'].items())[::-1][:8]}); print(pd.DataFrame(SALIENCY['by_family_feature']).T.round(3).to_string())
print('12. Complementarity @ budget', PB, '(VALIDATION | TEST):'); print(pd.DataFrame({f'{sp}|{s[:3]}': v for sp in COMPL for s, v in COMPL[sp].items()}).T.round(3).to_string())
print('13. Limitations: synthetic labels; 16-bar window only; possible generator-specific learning; selection+early stopping share validation; thresholds from ~13 genuine days; uncalibrated logits; saliency approximate; no spoofing/layering.')
print('14. FINAL VERDICT:', VERDICT)
print('=========================================================================')
if not ALL_PASS: raise AssertionError('validation failed; nothing published')

PASS T01 all required files exist
PASS T02 frozen sources unchanged (Layers 1, 2, 3 v1/v2, Autoencoder v1/v2, XGBoost v1)
PASS T03 protocol intact (hash matches) and frozen BEFORE any model; selection, LOFO and saliency only on train/validation; test unlocked last
PASS T04 preprocessing fitted on Layer 1 TRAIN bars only (independent numpy recompute; identical to the Autoencoder files)
PASS T05 comparator thresholds recomputed here equal the thresholds saved by the XGBoost / Autoencoder notebooks
PASS T06 the Transformer is causal (changing bars after t leaves the score at <= t unchanged) and reacts to later changes
PASS T07 input windows of clean bars are identical in genuine Layer 1 and Layer 3 (no injection leaks into negatives)
PASS T08 every fitting row is inside Layer 3 TRAIN; every early-stopping row inside VALIDATION; no test row used for fitting
PASS T09 clean-bar mask independently recomputed (plain loop on 6000 random bars) matches
PASS T10 every budget threshold recomputed f